# RAG Agent

The first agent in the agency. When a client uploads their company PDF, this agent:

## Part A -> Read Docs
## Part B -> Docs -> Chunks
## Part C -> Chunks -> VD
## Part D -> Ask the VD

Every PDF is kept in `knowledge-base/<company>/`, so next time we already have the customer's data.

In [1]:
import os
import glob
import shutil
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage

C:\Users\Asus\AppData\Local\Temp\ipykernel_18308\3135919000.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


In [2]:
MODEL = "gpt-5.4-nano"
db_name = "vector_db"
knowledge_base = "knowledge-base"   # knowledge-base/<company>/<file>.pdf
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", openai_api_key=openai_api_key)
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = ChatOpenAI(model=MODEL, openai_api_key=openai_api_key)


def company_id(company_name):
    # "Sunny Lane Cafe" -> "sunny_lane_cafe"  (used as folder name + doc_type)
    return company_name.strip().lower().replace(" ", "_")

# Part A

Read every company's PDFs. The folder name becomes the `doc_type`, so we know which company each document belongs to.

In [4]:
def load_documents():
    folders = glob.glob(f"{knowledge_base}/*")
    documents = []  # langchain document objects

    for folder in folders:
        if not os.path.isdir(folder):
            continue  # skip loose files; PDFs must be in knowledge-base/<company>/
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(folder, glob="**/*.pdf", loader_cls=PyPDFLoader)
        folder_docs = loader.load()

        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)

    return documents

### Just Inspect

In [5]:
folders = glob.glob(f"{knowledge_base}/*")
folders

['knowledge-base\\island_glow_skincare', 'knowledge-base\\sunny_lane_cafe']

In [6]:
documents = load_documents()
print(len(documents))
documents[:2]

2


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-07T09:15:22+05:00', 'author': 'Island Glow Skincare', 'keywords': '', 'moddate': '2026-11-07T09:15:22+05:00', 'subject': 'Company Profile & Brand Guide', 'title': 'Island Glow Skincare', 'trapped': '/False', 'source': 'knowledge-base\\island_glow_skincare\\island_glow_skincare.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'doc_type': 'island_glow_skincare'}, page_content='Island Glow Skincare - Company Profile & Brand Guide 2026\nPage 1\nIsland Glow Skincare\nCompany Profile & Brand Guide - 2026\nAbout us\nIsland Glow Skincare is a small-batch natural skincare brand from Colombo, Sri Lanka, founded in 2020 by\ncosmetic chemist Ruvini de Silva. Our products are made with Sri Lankan botanicals like virgin coconut oil,\nturmeric, gotu kola, sandalwood and Ceylon cinnamon, using Ayurveda-inspired recipes backed by modern\nformulation. We sell mainly online th

# Part B

Split the documents into chunks of 1000 characters, with 200 characters of overlap.

In [7]:
def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)
    return chunks

In [8]:
chunks = split_documents(documents)
print(len(chunks))
chunks[0]

5


Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-07T09:15:22+05:00', 'author': 'Island Glow Skincare', 'keywords': '', 'moddate': '2026-11-07T09:15:22+05:00', 'subject': 'Company Profile & Brand Guide', 'title': 'Island Glow Skincare', 'trapped': '/False', 'source': 'knowledge-base\\island_glow_skincare\\island_glow_skincare.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'doc_type': 'island_glow_skincare'}, page_content='Island Glow Skincare - Company Profile & Brand Guide 2026\nPage 1\nIsland Glow Skincare\nCompany Profile & Brand Guide - 2026\nAbout us\nIsland Glow Skincare is a small-batch natural skincare brand from Colombo, Sri Lanka, founded in 2020 by\ncosmetic chemist Ruvini de Silva. Our products are made with Sri Lankan botanicals like virgin coconut oil,\nturmeric, gotu kola, sandalwood and Ceylon cinnamon, using Ayurveda-inspired recipes backed by modern\nformulation. We sell mainly online thr

# Part C

Turn the chunks into vectors and save them in the Chroma vector database.

In [9]:
def save_to_vector_db(chunks):
    if os.path.exists(db_name):
        Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
    print(f"RAG AGENT: vector db built with {vectorstore._collection.count()} chunks", flush=True)
    return vectorstore

Part A + B + C together:

In [10]:
def build_vector_db():
    documents = load_documents()           # Part A
    chunks = split_documents(documents)    # Part B
    save_to_vector_db(chunks)              # Part C

# Part D

Ask the vector database about one company. We use a `filter` so we only get that company's chunks.

In [11]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable assistant for a creative digital marketing agency.
You are answering questions about our client company using their own documents.
Answer questions using only the given context. Do not use outside knowledge.
If the context does not contain the answer, say that you don't know.
Do not make up names, numbers, dates, or other details.
Context:
{context}
"""

In [12]:
def ask_company(company_name, question):
    print(f"RAG AGENT: asking about {company_name}: {question}", flush=True)
    vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 10, "filter": {"doc_type": company_id(company_name)}})

    docs = retriever.invoke(question)
    if not docs:
        return f"No data saved for {company_name}. Ask the user to upload their company PDF."

    text = []
    for doc in docs:
        text.append(doc.page_content)
    context = "\n".join(text)

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    system_message = SystemMessage(content=system_prompt)
    human_message = HumanMessage(content=question)
    response = llm.invoke([system_message, human_message])
    return response.content

# The tool the Planner calls

When the user uploads a PDF, the Planner calls `save_company_pdf`. It runs the entire thing:

copy the PDF into `knowledge-base/<company>/` -> Part A -> Part B -> Part C -> Part D (a short company profile)

In [13]:
def save_company_pdf(pdf_path, company_name):
    print(f"RAG AGENT: saving {pdf_path} for {company_name}", flush=True)
    folder = os.path.join(knowledge_base, company_id(company_name))
    os.makedirs(folder, exist_ok=True)
    if os.path.abspath(os.path.dirname(pdf_path)) != os.path.abspath(folder):
        shutil.copy(pdf_path, folder)

    build_vector_db()

    # Analyze the company so the planner knows who the customer is
    return ask_company(company_name, "Give a short profile of this company: what they sell, "
                                     "target audience, brand voice, brand colours, unique selling points "
                                     "and any marketing goals mentioned.")

## Try it

Put a company PDF next to this notebook and change the file name below.

In [14]:
profile = save_company_pdf("knowledge-base/sunny_lane_cafe/sunny_lane_cafe.pdf", "Sunny Lane Cafe")
print(profile)

RAG AGENT: saving knowledge-base/sunny_lane_cafe/sunny_lane_cafe.pdf for Sunny Lane Cafe
RAG AGENT: vector db built with 5 chunks
RAG AGENT: asking about Sunny Lane Cafe: Give a short profile of this company: what they sell, target audience, brand voice, brand colours, unique selling points and any marketing goals mentioned.
No data saved for Sunny Lane Cafe. Ask the user to upload their company PDF.


In [15]:
print(ask_company("Sunny Lane Cafe", "What are their best selling products?"))

RAG AGENT: asking about Sunny Lane Cafe: What are their best selling products?
No data saved for Sunny Lane Cafe. Ask the user to upload their company PDF.


# Visualize

Each dot is one chunk. Chunks from the same company get the same colour.

In [16]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
collection = vectorstore._collection
count = collection.count()
print(f"Total vectors in the collection: {count}")

Total vectors in the collection: 5


In [17]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
companies = sorted(set(doc_types))
palette = ['blue', 'green', 'red', 'orange', 'purple', 'brown']
colors = [palette[companies.index(t) % len(palette)] for t in doc_types]

In [18]:
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors) - 1))
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Company: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Convert to a python file

The Planner notebook imports this agent, so we convert the notebook into a `.py` file.
Cells tagged `test` (Just Inspect, Try it, Visualize) are removed, so importing the agent does not run them.

```
jupyter nbconvert --to script rag_agent.ipynb --TagRemovePreprocessor.remove_cell_tags=test
```